# 07 · Comparison with measured isotherms
Reproduces report section 5.7: 77 K isotherms (1–100 bar) of MOF-5, UiO-66, Zn-MOF-74 and HKUST-1, helium void fractions, conversion to excess and wt%, and the MOF-5 DOE-swing check against the measured 51.9 g/L (Ahmed et al. 2017).

**Runtime: about 2 hours on 4 cores.** **Attach:** the `mofh2_pipeline` dataset.

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = ['07_experiment']
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

### Helium void fractions (Widom insertion at 298 K)

In [ ]:
for m, u in [('IRMOF-1','1 1 1'), ('UIO-66','2 2 2'), ('Zn-DOBDC','2 2 4'), ('Cu-BTC','1 1 1')]:
    if not glob.glob(f'{WORK}/runs/he_{m}/Output/System_0/*.data'):
        !bash run_he.sh {m} "{u}" > /dev/null

### Run simulations: `07_experiment` (28 runs)

In [ ]:
!bash rq_any.sh queues/07_experiment.txt {MAX_HOURS}

### MOF-5 at 160 K/5 bar for the DOE swing (from stage 05, or run here if missing)

In [ ]:
if not glob.glob(f'{WORK}/runs_160/IRMOF-1_5/Output/System_0/*.data'):
    open('queues/07b.txt','w').write('IRMOF-1|160|5|1000|UFF_FHm2|1 1 1|runs_160/IRMOF-1_5\n')
    !bash rq_any.sh queues/07b.txt 2
!python exp_isotherms.py

### Compare with the original project and save outputs
The comparison prints mean and maximum differences between your reproduced numbers and the original ones. GCMC is stochastic, so expect small differences (typically under 1 g/L at 77 K/100 bar).

In [ ]:
!python compare_reference.py $PKG exp_gcmc_isotherms.csv

In [ ]:
!python stage_io.py save stage07_outputs
# Attach /kaggle/working/stage07_outputs.tgz (via 'Save Version' output) as input to later notebooks.